# Task 2 — Teacher Forcing & Training Strategies

## 1. The Training Input Dilemma

During training, the Decoder generates tokens step-by-step. To predict the token at step $t$, the decoder needs an **input token** representing what was generated at step $t-1$.

There are two primary ways to train this sequence:

![Teacher Forcing vs Free-Running](outputs/teacher-forcing.png)

### Option A: Free-Running Mode (No Teacher Forcing)
- The decoder feeds its **own prediction** from step $t-1$ as the input to step $t$.
- **The Problem**: If the model makes a mistake early in training (e.g., predicting *"He"* instead of *"I"* at step 1), all subsequent inputs are incorrect. The model wastes training time trying to generate coherent text based on a wrong prefix, leading to slow convergence and training instability. This is known as **compounding errors** or **error cascading**.

### Option B: Teacher Forcing Mode
- Regardless of what the decoder predicted at step $t-1$, we override it and feed the **actual ground-truth token** as the input to step $t$.
- **The Advantage**: The model is always trained on clean, correct prefixes. This drastically speeds up convergence and makes training highly stable.

---

## 2. Concrete Example Walkthrough

Let's illustrate both strategies with the translation pair:
- **Source**: *"je suis fatigué"*
- **Target**: *"i am tired"*
- **Target Tokens**: `["<SOS>", "i", "am", "tired", "<EOS>"]`

### Case 1: Free-Running Training (No Teacher Forcing)
At Step 1, the model is fed `<SOS>` and makes an incorrect prediction:
- **Step 1**: Input is `<SOS>` $ightarrow$ Model predicts **"he"** *(expected: "i")*
- **Step 2**: Input becomes the model's prediction **"he"** $ightarrow$ Model predicts **"is"** *(expected: "am")*
- **Step 3**: Input becomes the model's prediction **"is"** $ightarrow$ Model predicts **"small"** *(expected: "tired")*

Here, the mistake at Step 1 caused the entire sequence generation to cascade into an unrelated sentence (*"he is small"*). The model computes loss comparing output predictions to target tokens:
- Output: `["he", "is", "small"]` vs. Target: `["i", "am", "tired"]`
The gradients computed from this mismatched comparison are noisy, meaning the model struggles to learn the relationships between the original inputs and targets.

### Case 2: Teacher Forcing Training
At Step 1, the model is fed `<SOS>` and makes the same incorrect prediction:
- **Step 1**: Input is `<SOS>` $ightarrow$ Model predicts **"he"** *(Loss is computed comparing "he" to "i")*
- **Step 2**: We override the prediction and feed target token **"i"** $ightarrow$ Model predicts **"am"** *(expected: "am")*
- **Step 3**: We override and feed target token **"am"** $ightarrow$ Model predicts **"tired"** *(expected: "tired")*

Even though the model made a mistake at Step 1, we corrected it before Step 2. This keeps the model on track, ensuring it always receives valid sentence structures as inputs.

---

## 3. The Catch: Exposure Bias

While Teacher Forcing is brilliant for training, it introduces a critical problem called **Exposure Bias**:

During training, the model is exposed *only* to perfect ground-truth inputs. However, during inference (real-world testing), there is no ground-truth target available. The model must rely on its own predictions. 
If it makes a slight prediction error at step 1, it has never been trained to recover from its own mistakes, leading to a rapid degradation in output quality.

---

## 4. Scheduled Sampling

To bridge the gap between training (Teacher Forcing) and inference (Free-Running), we use **Scheduled Sampling**:

At the beginning of training, we set the `teacher_forcing_ratio = 1.0` (always use ground truth). As training progresses across epochs, we gradually decay this ratio toward `0.0`. This slowly forces the model to learn to handle its own predictions, correcting for exposure bias before deployment.
